In [2]:
import pandas as pd
import numpy as np
# import plotly.graph_objects as go
import os

# Set data directory
data_dir = '../data/'

In [3]:

# ... (your existing setup) ...
years = [2021, 2022, 2023, 2024, 2025]
jawa = ['JAWA BARAT', 'JAWA TENGAH', 'JAWA TIMUR', 'DKI JAKARTA', 'BANTEN', 'DI YOGYAKARTA'] # added DI Yogyakarta for completeness

processed_dfs = []

for year in years:
    file_path = os.path.join(data_dir, f'raw/ppo_indonesia_{year}.csv')
    
    # Read the file normally
    df = pd.read_csv(file_path, skiprows=4)
    
    # FIX: Select only the first 3 columns regardless of how many exist
    # column 0: Provinsi, column 1: Semester 1, column 2: Semester 2
    df = df.iloc[:, [0, 1, 2]]
    
    # Now we can safely rename because we only have 3 columns
    df.columns = ['Provinsi', 'S1', 'S2']
    
    # Clean up names (remove trailing/leading spaces which are common in BPS data)
    df['Provinsi'] = df['Provinsi'].str.strip()
    
    # Filter for Jawa
    df = df[df['Provinsi'].isin(jawa)].copy()
    
    # Convert to numeric
    df['S1'] = pd.to_numeric(df['S1'], errors='coerce')
    df['S2'] = pd.to_numeric(df['S2'], errors='coerce')
    
    # Fallback logic: Semester 2, if NaN then Semester 1
    df[f'{year}'] = df['S2'].fillna(df['S1'])
    
    processed_dfs.append(df[['Provinsi', f'{year}']])

final_comparison = processed_dfs[0]
for df in processed_dfs[1:]:
    final_comparison = pd.merge(final_comparison, df, on='Provinsi', how='outer')

print(final_comparison)
final_comparison.to_csv(data_dir + "clean/ppo_jawa_2021-2025.csv", index=False, encoding="utf-8")

        Provinsi   2021   2022   2023   2024  2025
0         BANTEN   6.04   5.89   6.00   5.57  5.35
1  DI YOGYAKARTA  11.20  10.64  10.27  10.11  9.99
2    DKI JAKARTA   4.67   4.61   4.44   4.14  4.03
3     JAWA BARAT   7.48   7.52   7.19   6.65  6.66
4    JAWA TENGAH  10.16  10.02   9.78   8.83  8.99
5     JAWA TIMUR   7.99   7.78   7.50   6.83  6.93


In [ ]:
import pandas as pd

# ==========================================
# COMBINE 2 DIFFERENT FILES (2025 ONLY)
# File 1 = poverty data
# File 2 = unemployment data
# ==========================================

# Change filenames here
poverty_file = data_dir + "raw/Persentase_Penduduk_Miskin_Menurut_Kabupaten_Kota_di_Jawa_Barat_2025.csv"
unemployment_file = data_dir + "raw/Tingkat_Pengangguran_Terbuka_Menurut_Kabupaten_Kota_2025.csv"
poverty_lvl_file = data_dir + "raw/Garis_Kemiskinan_Menurut_Kabupaten_Kota_2025.csv"


# ------------------------------------------
# Helper function to clean BPS-style files
# ------------------------------------------
def clean_bps_file(file_path, value_col_name):
    
    # Read all rows without header
    df = pd.read_csv(file_path, header=None)

    # Keep only rows with at least 2 columns
    df = df.iloc[:, :2]

    # Rename columns
    df.columns = ["region", value_col_name]

    df = df.dropna(subset=["region"])
    # Strip spaces
    df["region"] = df["region"].astype(str).str.strip()

    # Convert numeric
    df[value_col_name] = pd.to_numeric(df[value_col_name], errors="coerce")

    # Keep only valid numeric rows
    df = df.dropna(subset=[value_col_name])

    # Remove province total row
    df = df[df["region"] != "Provinsi Jawa Barat"]

    return df


# ------------------------------------------
# Load both files
# ------------------------------------------
df_poverty = clean_bps_file(poverty_file, "poverty_rate")
df_unemployment = clean_bps_file(unemployment_file, "unemployment_rate")
df_poverty_level = clean_bps_file(poverty_lvl_file, "poverty_level")

# ------------------------------------------
# Merge
# ------------------------------------------
df_final = pd.merge(df_poverty, df_unemployment, on="region", how="outer")
df_final = pd.merge(df_final, df_poverty_level, on="region", how="outer")


# ------------------------------------------
# Save clean combined file
# ------------------------------------------
df_final.to_csv(data_dir + "clean/jabar_2025_combined.csv", index=False, encoding="utf-8")


# ------------------------------------------
# Show result
# ------------------------------------------
print(df_final)


              region  poverty_rate  unemployment_rate  poverty_level
0            Bandung          6.04               6.68       468974.0
1      Bandung Barat          9.87               6.60       471101.0
2             Bekasi          4.36               8.78       698154.0
3              Bogor          6.26               7.69       531412.0
4             Ciamis          7.19               4.08       483644.0
5            Cianjur          9.82               6.17       482562.0
6            Cirebon         10.23               6.42       491604.0
7              Garut          9.39               6.54       407191.0
8          Indramayu         11.02               6.47       578169.0
9           Karawang          7.08               7.99       617901.0
10      Kota Bandung          3.78               7.22       644417.0
11       Kota Banjar          5.73               5.26       451277.0
12       Kota Bekasi          3.96               7.33       864601.0
13        Kota Bogor          5.89